_ExplainDB — Copyright (C) 2026 Prof. Dr. Jens Dittrich, Saarland University._  
_Licensed under the GNU Affero General Public License v3 (AGPL-3.0). See the `LICENSE` file._

# RAID Playground

Build two RAID arrays, **A** and **B**, side by side, fail disks and see which blocks are still there, what each
array costs, how fast it is and which disk failures it survives. All numbers come from the RAID cost model in
`system/storage/RAID/cost_model.py`, the placement of the blocks from `system/storage/RAID/block_assignment.py`.

- **Preset:** a single level (**RAID 0**, **RAID 1**, **RAID 5**) or two levels, named lower level first:
  **RAID 0+1** mirrors (RAID 1) two stripes (RAID 0), **RAID 1+0** stripes over mirrored pairs, **RAID 5+0** and
  **RAID 5+1** stripe or mirror over RAID 5 arrays.
- **Levels and members:** the **outer level** stripes or mirrors over its members. With an **inner level**, every
  member is an array of that level over **members per inner array** disks; with **none**, every member is a disk.
  RAID 0 and RAID 1 have at least two members, RAID 5 at least three, every level at most six. Once you change a
  level or a member count, the preset shows **own**.
- **Disks:** the disks are named **a**, **b**, **c**, ... and grouped by inner array in dashed boxes. A click on a
  disk marks it as **click marks** says: **failed** (a second click repairs it), **slow** or **fast**; an unmarked
  disk is fast. Changing a member count clears all marks of the array; changing only a level, or choosing a preset
  with the same member counts, keeps them, so you can compare the same failed disks under, e.g., RAID 0+1 and 1+0.
  Choosing RAID 5 may raise a member count to three, and adding or removing the inner level changes the number of
  disks; both clear the marks.
- **Blocks:** below each disk stand its first blocks, row by row, as many rows as the array has disks (at least four,
  at most twelve). A number is a logical data block, `P[0^1]` the parity (XOR) of the blocks 0 and 1, and `p[0^2]` the
  parity of an inner RAID 5 array over the other blocks of its row in that array. A block on a failed disk is orange
  if what it holds can still be read from a copy or rebuilt from the parity, and red if it is lost (see *What the
  model assumes* below). Every block is checked on its own, at both levels; a block that one level can read or rebuild
  can help the other level to rebuild another one.
- **Verdict:** whether all data can still be read with the failed disks, as the cost model decides it (see *What
  the model assumes* below). A note says so where every block can still be read on its own although the model says
  that data is lost.
- **Numbers:** below each array: its disks, the storage blow-up (raw capacity divided by usable capacity) and the
  usable share of the raw capacity, the sequential read and write throughput, the random reads and writes per second
  (IOPS), and of all combinations of one, two and three failed disks how many the array survives. The rows of A and B
  stand at the same height, and the better value of each row is in bold; for the failures the larger share counts,
  so "4 of 6" beats "9 of 15".
- **compare with B:** switch it off to hide B; A then takes the full width.
- **Device types:** the four values of a slow and of a fast disk, for both arrays; a fast disk starts twice as fast
  as a slow one.

In [ ]:
import sys, pathlib

# Put the repo root (the directory containing `system/`) on sys.path so
# `from system...` works whether this notebook is opened from its own
# folder or from the repo root.
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "system").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

In [ ]:
from system.playground.raid_widgets import RaidPlayground

RaidPlayground().widget

## Things to try

- **0+1 vs 1+0:** A and B start as RAID 0+1 and RAID 1+0 over four disks, with the same numbers except for the
  pairs of failed disks they survive. Fail disks a and d in both: RAID 1+0 still reads all data. RAID 0+1 loses
  data by the model, although every block is still on b or c, as its orange blocks show; read the note below its
  verdict. Then fail a and c instead: which blocks turn red?
- **Group slow and fast disks:** in A (RAID 0+1), make disks a and b slow, then instead disks a and c. Why does the
  sequential read throughput change, although the same number of disks is slow? Try the same in B (RAID 1+0).
- **RAID 5 writes:** set A to RAID 5 over four disks and B to RAID 1+0. Both read randomly at the same rate, but RAID 5
  writes fewer blocks per second: a small write reads the old data and parity block and writes both anew. What does
  RAID 5 gain in exchange?
- **RAID 5 rows:** set A to RAID 5 and follow the parity `P[…]` from row to row: it moves on by one disk per row, so
  that every disk holds data and parity. Fail one disk: its blocks turn orange, as the parity rebuilds them. Fail a
  second disk: both turn red. Compare RAID 0 and RAID 1.
- **Parity in nested arrays:** set A to RAID 5+0. The stripe puts the even blocks on the first RAID 5 array and the
  odd ones on the second, and each array adds its own parity `p[…]`. Which pairs of failed disks does it survive?
- **Wide arrays:** switch **compare with B** off, so that A takes the full width, and give A more members, e.g.
  RAID 5+1 over two RAID 5 arrays of six disks each.

## What the model assumes

- An array member that loses any data counts as failed as a whole; the model never combines the surviving parts of
  two members, as a typical RAID 0+1 controller does not. In RAID 0+1 with the stripes (a, b) and (c, d), failing
  disks a and d therefore loses data, although every block still exists once (on b and c). The blocks below the
  disks are checked one by one and show the difference: they stay orange.
- A block on a failed disk is rebuilt row by row, as RAID controllers do: read from a copy, or computed as the XOR
  of the other blocks of a RAID 5 row, and a block obtained this way can help to rebuild the next one. In rare cases
  of RAID 5 over RAID 5 with many failed disks, a block could still be computed from several rows together; the
  playground shows it red.
- All performance numbers assume that every disk works: a failed disk does not make the array slower (no degraded
  mode, no rebuild).